<a href="https://colab.research.google.com/github/reknahs/DeRozan/blob/filestuff/Copy_of_azure_anonymization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install openai
!pip install bs4
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.1 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [10]:
%%capture
from google.colab import userdata
API_VERSION = userdata.get('API_VERSION')
API_KEY = userdata.get('API_KEY')
AZURE_ENDPOINT = userdata.get('AZURE_ENDPOINT')
DEPLOYMENT_NAME = userdata.get('DEPLOYMENT_NAME')

In [4]:
import re

def remove_links(text):
    # Regular expression pattern to match URLs
    url_pattern = re.compile(r'https?://\S+|www\.\S+')

    # Replace all matches with an empty string
    text_without_links = re.sub(url_pattern, '', text)

    return text_without_links

In [5]:
import email
from email import policy
from email.parser import BytesParser

def get_email_body(email_path):
  # Read the .eml file
  with open(email_path, 'rb') as f:
      msg = BytesParser(policy=policy.default).parse(f)

  # Extract subject
  subject = msg['subject']
  print('Subject:', subject)

  # Extract body (both plain text and HTML)
  body = ""
  if msg.is_multipart():
      for part in msg.iter_parts():
          content_type = part.get_content_type()
          content_disposition = str(part.get('Content-Disposition'))

          # Look for plain text parts
          if content_type == 'text/plain' and 'attachment' not in content_disposition:
              body = part.get_payload(decode=True).decode(part.get_content_charset())
              break
          # Look for HTML parts
          elif content_type == 'text/html' and 'attachment' not in content_disposition:
              body = part.get_payload(decode=True).decode(part.get_content_charset())
              break
  else:
      # If the message isn't multipart, the payload is simple text
      body = msg.get_payload(decode=True).decode(msg.get_content_charset())
  return subject+"\n\n"+body

In [6]:
from bs4 import BeautifulSoup

# remove all tags
def remove_tags(html):

    # parse html content
    soup = BeautifulSoup(html, "html.parser")

    for data in soup(['style', 'script']):
        # Remove tags
        data.decompose()

    # return data by retrieving the tag content
    initial = re.sub(r'[\u00A0\u200C]+', '\n', ' '.join(soup.stripped_strings))
    return re.sub(r'\n{3,}', '\n', initial)


In [11]:
import os
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=API_KEY,
    api_version=API_VERSION,
    azure_endpoint = AZURE_ENDPOINT
    )

deployment_name= DEPLOYMENT_NAME

In [12]:
# for chat completion requests
def completion(messages, tools=None, tool_choice=None, model=deployment_name):
    try:
        response = client.chat.completions.create(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice=tool_choice,
            temperature=0
        )
        return response
    except Exception as e:
        print("Unable to generate ChatCompletion response")
        print(f"Exception: {e}")
        return e

In [13]:
# Send a completion call to generate an answer
def anonymize(email):
  content = "Your job is to output a given email but after anonymizing any PII. PII is anything that can give away someone's identity."
  prompt = '''
  Replace ALL instances of the following for the email recipient:
  -Name
  -Billing/shipping address
  -Phone number
  -Email address
  -Order number
  -Any portion of credit card number

  Replace the above with fake info. Never delete without replacing. Never add additional info. Do not change prices or product names.
  If no PII is detected, simply return the original email.
  Example:

  Input:
  Hi James,

  We'd like to confirm your order (Order number: 6823) to the 8523 Main St. San Francisco, CA 92356.
  Paid with Mastercard ending in 9823.

  Example of your output:
  Hi Stephen,

  We'd like to confirm your order (Order number: 9866) to the 1323 Oak St. Orlando, FL 12235.
  Paid with Mastercard ending in 0912.

  Here is the email:

  '''+email
  messages = messages=[{"role": "system", "content": content},
                       {"role": "user", "content": prompt}]
  return completion(messages)

In [14]:
# read in emails from dataset
import random

directory = "/content/drive/MyDrive/mailbox/"

# remove duplicate emails from the list
def remove_copies(file_names):
    pattern = re.compile(r'-\d{1,2}\.eml$')

    return [s for s in file_names if not pattern.search(s)]

file_names = remove_copies(os.listdir(directory))

# index 718 has an eml file that doesn't work
file_names.pop(718)

'.DS_Store'

In [15]:
# test
num = random.randint(0, 863)
print(num)
processed = remove_tags(remove_links(get_email_body(directory+file_names[num])))
print(processed)

596
Subject: Confirm email address
Confirm email address

    Confirm email address                          Account             Hi
there, 

 We're almost done. At Bitdefender, we are all about safety. You just need
to verify your Bitdefender Central account, and you are good to go. 

 Please click on the button below to verify your account.          Verify
now
<
         In case you are wondering, this step is necessary to ensure that
no one can hijack your account. 
*It's a one-time thing: you just need to verify your account once, and
you'll be all set.*

 Thanks for choosing Bitdefender!             *Manage security from your
phone *      *Download Bitdefender Central app*
<
<          
               Please do not reply to this email. This message was sent
from a notification-only address that is not monitored. Instead, please use
the following link to contact our Bitdefender Customer Care
representatives:
  
 
 *Customer Care* < |
*Terms of Service* <

 
 * Sent by: Bitdefender, 

In [16]:
# test
print(anonymize(processed).choices[0].message.content)

Confirm email address

    Confirm email address                          Account             Hi
there, 

 We're almost done. At Bitdefender, we are all about safety. You just need
to verify your Bitdefender Central account, and you are good to go. 

 Please click on the button below to verify your account.          Verify
now
<
         In case you are wondering, this step is necessary to ensure that
no one can hijack your account. 
*It's a one-time thing: you just need to verify your account once, and
you'll be all set.*

 Thanks for choosing Bitdefender!             *Manage security from your
phone *      *Download Bitdefender Central app*
<
<          
               Please do not reply to this email. This message was sent
from a notification-only address that is not monitored. Instead, please use
the following link to contact our Bitdefender Customer Care
representatives:
  
 
 *Customer Care* < |
*Terms of Service* <

 
 * Sent by: Bitdefender, Bucharest, District 6, 15A Orhideel

In [17]:
schema = [
  {
    "type": "function",
    "function": {
      "name": "orders_details_email_header",
      "description": "Send important details of an email to a client",
      "parameters": {
        "type": "object",
        "properties": {
          "SenderDomain": {
            "type": "string",
            "description": "The domain name (without the subdomain) of the sender of the email. If the email was sent on the behalf of a third party, this should be the domain name of the third party."
          },
          "OrderId": {
            "type": "string",
            "description": "The order id/order number of the order the email contains"
          },
          "OrderDate": {
            "type": "string",
            "description": "The date that the order the email contains was sent, in MM/DD/YYYY format"
          },
          "OrderCurrency": {
            "type": "string",
            "description": "The currency used in the order the email contains, in ISO 4217 3-letter currency code format."
          },
          "OrderLineItems": {
            "type": "string",
            "description": "A list containing any and all line items contained in the order from the email. Each line item should follow the format of 'Item Name: Item Quantity x Unit Price'. The unit price be a number only, and not include a currency code."
          },
          "OrderTax": {
            "type": "string",
            "description": "The tax paid for the order the email contains. This value should be a number only, and not include a currency code or symbol."
          },
          "OrderTotal": {
            "type": "string",
            "description": "The total amount paid for the order the email contains. This value should be a number only, and not include a currency code or symbol"
          },
          "PaymentMethod": {
            "type": "string",
            "description": "The general method or scheme used for performing the payment associated with the order. Examples are 'Visa', 'MasterCard', 'PayPal' etc."
          },
          "PaymentInstrument": {
            "type": "string",
            "description": "The specific instance of a Payment Method used for the order. Examples are 'Visa card ending in 1234', 'MasterCard ending in 2345', 'PayPal account with username john.doe@example.com'"
          },
          "ShippingMethod": {
            "type": "string",
            "description": "The shipping method used for the order"
          },
          "ExpectedDeliveryDate": {
            "type": "string",
            "description": "The expected delivery date of the order in MM/DD/YYYY format"
          },
          "SchemaVersion": {
            "type": "string",
            "description": "This value should always be 1.0"
          }
        },
        "required": [
          "SchemaVersion"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "offers_coupons_details_email_header",
      "description": "Send important details of an email to a client",
      "parameters": {
        "type": "object",
        "properties": {
          "SenderDomain": {
            "type": "string",
            "description": "The domain name (without the subdomain) of the sender of the email. If the email was sent on the behalf of a third party, this should be the domain name of the third party."
          },
          "OfferDescription": {
            "type": "string",
            "description": "The description of the offer contained in the email, containing the item(s) the offer is for and the coupon code if present"
          },
          "ExpirationDate": {
            "type": "string",
            "description": "The expiration date of the offer contained in the email, in MM/DD/YYYY format"
          },
          "SchemaVersion": {
            "type": "string",
            "description": "This value should always be 1.0"
          }
        },
        "required": [
          "SchemaVersion"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "refunds_details_email_header",
      "description": "Send important details of an email to a client",
      "parameters": {
        "type": "object",
        "properties": {
          "SenderDomain": {
            "type": "string",
            "description": "The domain name (without the subdomain) of the sender of the email. If the email was sent on the behalf of a third party, this should be the domain name of the third party."
          },
          "OrderId": {
            "type": "string",
            "description": "The order id/order number of the order the email contains a refund for"
          },
          "RefundDate": {
            "type": "string",
            "description": "The date that the refund contained in the email was send, in MM/DD/YYYY format"
          },
          "RefundCurrency": {
            "type": "string",
            "description": "The currency used in the refund the email contains, in ISO 4217 3-letter currency code format."
          },
          "RefundTotal": {
            "type": "string",
            "description": "The total amount refunded from the refund the email contains. This value should be a number only, and not include a currency code or symbol"
          },
          "RefundPaymentInstrument": {
            "type": "string",
            "description": "The specific instance of a Payment Method the refund was sent to. Examples are 'Visa card ending in 1234', 'MasterCard ending in 2345', 'PayPal account with username john.doe@example.com'"
          },
          "SchemaVersion": {
            "type": "string",
            "description": "This value should always be 1.0"
          }
        },
        "required": [
          "SchemaVersion"
        ]
      }
    }
  },
  {
    "type": "function",
    "function": {
      "name": "cancellations_details_email_header",
      "description": "Send important details of an email to a client",
      "parameters": {
        "type": "object",
        "properties": {
          "SenderDomain": {
            "type": "string",
            "description": "The domain name (without the subdomain) of the sender of the email. If the email was sent on the behalf of a third party, this should be the domain name of the third party."
          },
          "OrderId": {
            "type": "string",
            "description": "The order id/order number of the order the email contains a cancellation for"
          },
          "CancellationDate": {
            "type": "string",
            "description": "The date that the cancellation contained in the email was send, in MM/DD/YYYY format"
          },
          "RefundCurrency": {
            "type": "string",
            "description": "The currency used in the refund the email contains, in ISO 4217 3-letter currency code format."
          },
          "RefundTotal": {
            "type": "string",
            "description": "The total amount refunded from the refund the email contains. This value should be a number only, and not include a currency code or symbol"
          },
          "RefundPaymentInstrument": {
            "type": "string",
            "description": "The specific instance of a Payment Method the refund was sent to, if present. Examples are 'Visa card ending in 1234', 'MasterCard ending in 2345', 'PayPal account with username john.doe@example.com'"
          },
          "RefundDate": {
            "type": "string",
            "description": "The date that the refund contained in the email is estimated to be send if present, in MM/DD/YYYY format"
          },
          "SchemaVersion": {
            "type": "string",
            "description": "This value should always be 1.0"
          }
        },
        "required": [
          "SchemaVersion"
        ]
      }
    }
  }
]

In [32]:
# for classifying emails
def classify(email):
  messages = []
  user = '''
  Classify the given email into one of the following categories(the one it fits best) If stuck between two categories, choose the most accurate:
  1. Order (¤) - The user MUST have already bought somethng
  2. Offer, Coupon, or Sale (†) - There MUST be a discount, coupon, or sale
  3. Refund (æ)
  4. Cancellation (⁍)
  5. Receipt or Invoice (¶)
  6. Shipping and Tracking Email (§)
  7. Newsletter (¦)

  Output BOTH the category of the email and the symbol it corresponds to only.

  Here is the email:
  '''
  messages.append({"role": "system", "content": "You are an AI assistant that helps users classify emails. "})
  messages.append({"role": "user", "content": user+" "+email})
  chat_response = completion(messages)
  return chat_response.choices[0].message.content

# for specifying class
def specify(string):
  if "¤" in string:
    return 1
  elif "†" in string:
    return 2
  elif "æ" in string:
    return 3
  elif "⁍" in string:
    return 4
  elif "¶" in string:
    return 5
  elif "§" in string:
    return 6
  elif "¦" in string:
    return 7
  else:
    print("Mistake found")
    return 7

In [19]:
# extracts necessary information from email given the labeling of the email
def extract(email, category):
  if category > 4:
    return "N/A"
  messages = []
  messages.append({"role": "system", "content": "You extract necessary information from emails."})
  messages.append({"role": "user", "content": 'Extract the necessary information from this email. If you cannot find something, just write the None type for it: '+email})
  options = ["orders_details_email_header", "offers_coupons_details_email_header", "refunds_details_email_header", "cancellations_details_email_header"]
  chat_response = completion(messages, tools=schema, tool_choice={"type": "function", "function": {"name": options[category-1]}})
  return chat_response.choices[0].message.tool_calls[0].function.arguments

In [35]:
# got labels for each email
# labels = []
# for i in range(0, len(file_names)):
#   if i % 100 == 0: print(i)
#   processed = remove_tags(remove_links(get_email_body(directory+file_names[i])))
#   label = specify(classify(processed))
#   labels.append(specify(classify(processed)))

# labels for future
labels = [7, 1, 2, 7, 7, 7, 7, 2, 7, 2, 2, 7, 2, 7, 7, 7, 7, 7, 2, 7, 7, 7, 7, 7, 7, 2, 7, 2, 2, 7, 7, 7, 7, 2, 2, 2, 2, 7, 2, 2, 2, 7, 1, 7, 7, 7, 7, 7, 7, 7, 7, 2, 2, 7, 1, 2, 2, 2, 7, 2, 2, 2, 2, 2, 2, 7, 7, 2, 7, 7, 2, 2, 7, 2, 7, 2, 7, 2, 7, 7, 7, 2, 7, 7, 7, 2, 7, 7, 7, 7, 2, 7, 1, 2, 7, 2, 2, 7, 7, 7, 7, 1, 7, 7, 7, 7, 7, 2, 7, 7, 7, 7, 2, 7, 1, 7, 7, 2, 2, 1, 2, 2, 2, 7, 7, 2, 7, 1, 7, 2, 7, 7, 7, 2, 2, 1, 7, 2, 2, 7, 7, 2, 7, 7, 2, 7, 7, 7, 2, 2, 2, 7, 7, 7, 2, 7, 2, 2, 2, 7, 2, 2, 2, 2, 2, 1, 2, 6, 1, 2, 2, 2, 2, 2, 2, 7, 2, 7, 1, 1, 7, 7, 1, 2, 2, 2, 2, 7, 2, 7, 7, 7, 7, 2, 2, 2, 2, 7, 2, 7, 1, 7, 7, 7, 7, 2, 2, 7, 2, 2, 2, 2, 2, 2, 7, 7, 7, 2, 1, 2, 2, 2, 7, 2, 7, 7, 7, 2, 7, 2, 7, 2, 7, 7, 7, 7, 7, 2, 1, 7, 7, 2, 2, 2, 6, 7, 1, 2, 7, 1, 7, 7, 7, 2, 7, 2, 7, 1, 2, 2, 7, 7, 7, 7, 7, 7, 7, 2, 7, 1, 7, 2, 7, 7, 7, 2, 2, 2, 7, 7, 7, 2, 7, 7, 2, 7, 2, 7, 7, 2, 7, 2, 7, 1, 7, 7, 2, 2, 7, 7, 7, 7, 7, 2, 7, 2, 2, 7, 7, 2, 7, 7, 7, 2, 7, 7, 2, 7, 7, 7, 7, 1, 7, 2, 2, 1, 2, 2, 7, 7, 7, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 2, 7, 7, 7, 7, 2, 2, 2, 7, 2, 7, 7, 7, 7, 7, 2, 7, 7, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 2, 4, 7, 7, 2, 2, 7, 2, 2, 7, 1, 7, 2, 2, 2, 2, 7, 7, 1, 7, 2, 7, 7, 7, 2, 2, 7, 7, 7, 7, 2, 7, 7, 1, 7, 2, 7, 7, 7, 7, 7, 2, 7, 7, 7, 2, 2, 2, 2, 2, 2, 1, 2, 7, 2, 2, 2, 7, 2, 7, 1, 7, 2, 7, 1, 7, 7, 7, 7, 2, 2, 7, 7, 7, 7, 7, 2, 7, 7, 2, 1, 7, 7, 7, 7, 2, 2, 7, 7, 2, 1, 7, 7, 2, 2, 2, 2, 2, 2, 1, 2, 7, 2, 7, 2, 2, 7, 2, 2, 7, 7, 7, 7, 7, 2, 2, 2, 7, 7, 7, 2, 2, 7, 7, 7, 1, 7, 7, 7, 2, 2, 7, 2, 7, 6, 7, 7, 7, 7, 7, 2, 7, 7, 1, 7, 7, 2, 2, 7, 7, 2, 2, 7, 7, 2, 2, 7, 1, 7, 2, 2, 7, 2, 2, 7, 1, 2, 2, 2, 2, 2, 7, 7, 7, 2, 2, 2, 2, 2, 2, 7, 7, 7, 7, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 2, 7, 7, 2, 7, 2, 7, 7, 1, 2, 7, 7, 7, 7, 4, 2, 7, 7, 7, 2, 2, 7, 2, 7, 6, 7, 7, 2, 7, 7, 7, 2, 7, 2, 1, 7, 7, 7, 7, 7, 7, 2, 7, 2, 2, 7, 2, 7, 7, 7, 7, 7, 2, 7, 7, 7, 2, 7, 1, 2, 2, 7, 2, 2, 7, 2, 7, 7, 2, 7, 2, 7, 7, 7, 7, 2, 7, 1, 7, 2, 2, 7, 1, 7, 7, 7, 1, 2, 2, 7, 7, 7, 2, 7, 2, 7, 7, 7, 7, 1, 7, 2, 7, 1, 7, 7, 2, 2, 7, 2, 7, 7, 7, 7, 7, 2, 1, 7, 1, 2, 1, 7, 2, 1, 2, 2, 2, 7, 1, 7, 7, 2, 7, 7, 2, 2, 7, 2, 7, 7, 2, 1, 2, 2, 2, 2, 2, 2, 2, 7, 2, 7, 2, 7, 7, 7, 7, 7, 7, 2, 7, 7, 2, 7, 2, 7, 7, 7, 7, 2, 7, 7, 7, 2, 7, 7, 7, 1, 7, 7, 7, 3, 2, 7, 7, 2, 2, 2, 7, 2, 7, 2, 7, 7, 1, 2, 2, 7, 2, 7, 2, 7, 2, 7, 7, 7, 7, 2, 7, 2, 7, 7, 1, 7, 7, 7, 2, 7, 2, 7, 7, 1, 2, 7, 2, 7, 2, 7, 7, 7, 2, 2, 2, 7, 7, 7, 2, 2, 7, 2, 7, 7, 7, 7, 7, 7, 2, 1, 7, 7, 7, 2, 2, 2, 7, 7, 2, 1, 7, 7, 2, 7, 7, 2, 7, 7, 7, 7, 2, 2, 2, 2, 2, 2, 7, 7, 2, 2, 1, 7, 2, 7, 2, 2, 7, 7, 2, 7, 7, 7, 1]

In [21]:
# 38/863 (4.4%) of all emails had no symbols found (default is newsletter)

In [36]:
#indices for each label
orders = [i for i in range(len(labels)) if labels[i] == 1]
offers = [i for i in range(len(labels)) if labels[i] == 2]
refunds = [i for i in range(len(labels)) if labels[i] == 3]
cancellations = [i for i in range(len(labels)) if labels[i] == 4]
receipts = [i for i in range(len(labels)) if labels[i] == 5]
shipping = [i for i in range(len(labels)) if labels[i] == 6]
newsletters = [i for i in range(len(labels)) if labels[i] == 7]

In [37]:
# number of each label in the dataset
print(len(orders), len(offers), len(refunds), len(cancellations), len(receipts), len(shipping), len(newsletters))

61 319 1 2 0 5 475


In [39]:
# want to assess accuracy of labeler so need random subsets of each list
orders_sample = random.sample(orders, 20)
offers_sample = random.sample(offers, 20)
refunds_sample = random.sample(refunds, 1) # edge case
cancellations_sample = random.sample(cancellations, 2) # edge case
# receipts_sample = random.sample(receipts, 20)
shipping_sample = random.sample(shipping, 5) # edge case
newsletters_sample = random.sample(newsletters, 20)

In [41]:
# results for labeler accuracy test
# orders: 18/20 or 19/20 (indices 685 - HTML issue and 704 - ambiguous)
# offers: 20/20 - very similar to newsletters, hard to gauge difference
# refunds: 1/1 - key word is that money has been given back
# cancellations: 2/2 - both emails emphasize money hasn't been refunded
# receipts: 0/0 - none found
# shipping and tracking: 0/11 - most are newsletters/offers that emphasize discounted shipping or fast shipping
# newsletters: 20/20 - very similar to offers, hard to gauge difference

# for i in orders_sample:
#   print(i)
#   print(remove_tags(remove_links(get_email_body(directory+file_names[i]))))
#   print("[----------------------------------------------------------------------]")

In [44]:
# results for extraction accuracy test
# orders: 18/18 (2 mislabels)
# offers: 16/20 (all errors were making up expiration dates)
# refunds: 1/1
# cancellations: 2/2

for i in cancellations_sample:
  print(i)
  label = labels[i]
  email = remove_tags(remove_links(get_email_body(directory+file_names[i])))
  print(email)
  print(extract(email, label))
  print("[----------------------------------------------------------------------]")

586
Subject: Order #1677 has been canceled
Order #1677 has been canceled

Your order has been canceled

curativelifestyle

Order #1677

----------------------------
Your order has been canceled
----------------------------

Order #1677 was canceled at your request and your payment
has not yet been refunded.

Removed Items
-------------

Zesty Elixir: Ginger Ale × 1

Refunded

$1.00

Subtotal

$1.00

Shipping

$4.90

Taxes

$0.00

Total

$5.90 USD

If you have any questions, reply to this email or
contact us at orders@curativelifestyle.com
{
    SenderDomain: "curativelifestyle.com",
    OrderId: "1677",
    CancellationDate: None,
    RefundCurrency: "USD",
    RefundTotal: "5.90",
    RefundPaymentInstrument: None,
    RefundDate: None,
    SchemaVersion: "1.0"
}

[----------------------------------------------------------------------]
375
Subject: Order #1700 has been canceled
Order #1700 has been canceled

Your order has been canceled

curativelifestyle

Order #1700

---------------